# Movie Genre Model Training
COCO 2024-2의 줄거리 기반 TF-IDF + Random Forest 학습 기록입니다.
데이터·모델 경로를 저장소 구조에 맞게 정리했습니다. 기존 출력은 과거 실행 기록이며 재실행 결과가 아닙니다.
전체 데이터 벡터화 후 train/test를 나누는 원본 구조를 보존했습니다. 엄격한 평가에서는 학습 데이터로만 벡터라이저를 fit해야 합니다.

In [24]:
import pandas as pd

# 데이터 불러오기
from pathlib import Path
ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    raise FileNotFoundError('Run this notebook from the repository root.')
(ROOT / 'models').mkdir(exist_ok=True)
data_path = ROOT / 'data' / 'movie_data_cleaned.csv' 
data = pd.read_csv(data_path)

In [25]:
data.head()

,id,movie_name,synopsis,genre
0,44978,Super Me,A young scriptwriter starts bringing valuable ...,fantasy
1,50185,Entity Project,A director and her friends renting a haunted h...,horror
2,34131,Behavioral Family Therapy for Serious Psychiat...,This is an educational video for families and ...,family
3,78522,Blood Glacier,Scientists working in the Austrian Alps discov...,scifi
4,2206,Apat na anino,Buy Day - Four Men Widely - Apart in Life - By...,action


In [26]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 53990 entries, 0 to 53989
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          53990 non-null  int64 
 1   movie_name  53990 non-null  object
 2   synopsis    53990 non-null  object
 3   genre       53990 non-null  object
dtypes: int64(1), object(3)
memory usage: 1.6+ MB


In [27]:
# label encode the genre
from sklearn.preprocessing import LabelEncoder
label_encoder = LabelEncoder()
# fit_transform: 라벨 인코딩 후 변환
data['genre_encoded'] = label_encoder.fit_transform(data['genre'])


In [28]:
data.head()

,id,movie_name,synopsis,genre,genre_encoded
0,44978,Super Me,A young scriptwriter starts bringing valuable ...,fantasy,4
1,50185,Entity Project,A director and her friends renting a haunted h...,horror,5
2,34131,Behavioral Family Therapy for Serious Psychiat...,This is an educational video for families and ...,family,3
3,78522,Blood Glacier,Scientists working in the Austrian Alps discov...,scifi,8
4,2206,Apat na anino,Buy Day - Four Men Widely - Apart in Life - By...,action,0


In [40]:
import numpy as np
num = np.random.randint(0, len(data))
print(data['synopsis'][num])

A policewoman becomes entangled in Japan's S&M underground, while investigating a gruesome murder.


In [41]:
print(data['genre'][num])

mystery


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Text Vectorization using TF-IDF
# 문서 내의 단어의 중요성을 평가하기 위해 사용되는 통계적 방법
# 특정 단어가 한 문서 내에서 얼마나 자주 등장하는지와, 얼마나 일반적이지 않은 단어인지를 판별해서 텍스트에서 의미 추출
tfidf_vectorizer = TfidfVectorizer(max_features=5000, stop_words='english')
# fit_transform: 벡터화 후 변환
X = tfidf_vectorizer.fit_transform(data['synopsis']) # input
y = data['genre_encoded'] # output

In [6]:
from sklearn.model_selection import train_test_split

# testing set, training set으로 나누기
# X_train: Training data features (TF-IDF vectors of movie synopses)
# X_test: Test data features (TF-IDF vectors of movie synopses) 
# y_train: Training data labels (encoded genre labels)
# y_test: Test data labels (encoded genre labels)
# test_size=0.2: 20% of data used for testing, 80% for training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [31]:
from sklearn.ensemble import RandomForestClassifier

# randomforest classifier 사용해서 분류
rf_model_simplified = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42)
# fit: 모델 학습
rf_model_simplified.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, random_state=42)

In [32]:
from sklearn.metrics import accuracy_score
# use test set으로 예측
y_pred = rf_model_simplified.predict(X_test)

# 정확도 확인
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy:.2f}")


Accuracy: 0.29


In [42]:
import joblib

# Save the simplified Random Forest model and TF-IDF vectorizer
joblib.dump(rf_model_simplified, ROOT / 'models' / 'best_rf_model_simplified.pkl')
joblib.dump(tfidf_vectorizer, ROOT / 'models' / 'tfidf_vectorizer.pkl')
joblib.dump(label_encoder, ROOT / 'models' / 'label_encoder.pkl')

['label_encoder.pkl']